# TAHAP 06 — Chatbot Pipeline Integration

**Project:** EduPath Career Assistant  
**Judul:** Chatbot Rekomendasi Program Studi S1 dan Roadmap Belajar untuk Siswa SMA/SMK/MA Berbasis NLP, Intent Classification, dan Recommender System.

Tahapan ini mengintegrasikan hasil:
1. **Tahap 04** — Intent Classification Modeling.
2. **Tahap 05** — Program Studi Recommender System & Roadmap Mapping.

Output utama notebook:
- Intent routing.
- Fungsi `chatbot_response()`.
- Pengujian beberapa skenario input.
- Penyimpanan report hasil integrasi chatbot.
- Checklist file hasil untuk validasi sebelum push ke GitHub.

## 1. Tujuan Tahap 06

Tahap 06 bertujuan menggabungkan model intent classification dan recommender system menjadi satu pipeline chatbot sederhana. Pipeline ini tidak hanya memprediksi intent user, tetapi juga menentukan tindakan yang sesuai, seperti memberikan rekomendasi program studi, roadmap belajar, prospek karier, skill awal, informasi program studi, atau respons fallback.

Secara akademik, tahap ini menunjukkan penerapan **Text Mining**, **Classification**, **Content-Based Recommender System**, dan **Rule-Based Routing** dalam satu alur sistem yang dapat diuji secara end-to-end.

## 2. Desain Pipeline Chatbot

Alur kerja chatbot pada tahap ini adalah sebagai berikut:

1. User memasukkan teks bebas dalam Bahasa Indonesia formal, non-formal, Jawa umum, atau Sunda umum.
2. Sistem melakukan text cleaning, normalisasi kata, tokenisasi sederhana, dan filtering.
3. Model intent classification memprediksi intent dari input user.
4. Rule-based routing membantu memperbaiki arah respons untuk keyword yang sangat eksplisit.
5. Berdasarkan intent akhir, sistem menjalankan modul respons:
   - `rekomendasi_prodi`
   - `roadmap_belajar`
   - `prospek_karier`
   - `skill_awal`
   - `info_program_studi`
   - `klarifikasi_minat`
   - `sapaan`
   - `fallback`
6. Sistem menghasilkan respons chatbot dan menyimpan hasil uji ke folder `reports/stage06/`.

In [1]:
# ============================================================
# 1. Import Library
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import re
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

print("Library berhasil di-load.")

Library berhasil di-load.


In [2]:
# ============================================================
# 2. Konfigurasi Path Project
# ============================================================

CURRENT_DIR = Path.cwd()

# Jika notebook dijalankan dari folder notebooks, project root adalah parent folder.
# Jika notebook dijalankan dari root project, gunakan current directory.
if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Fallback untuk paket ZIP hasil export jika folder bernama data-processed.
if not DATA_DIR.exists() and (PROJECT_ROOT / "data-processed").exists():
    DATA_DIR = PROJECT_ROOT / "data-processed"

MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports" / "stage06"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR    :", DATA_DIR)
print("MODEL_DIR   :", MODEL_DIR)
print("REPORT_DIR  :", REPORT_DIR)

PROJECT_ROOT: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant
DATA_DIR    : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\processed
MODEL_DIR   : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\models
REPORT_DIR  : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage06


In [3]:
# ============================================================
# 3. Load Dataset Processed
# ============================================================

dataset_paths = {
    "intent": DATA_DIR / "intent_dataset_processed.csv",
    "program": DATA_DIR / "program_studi_s1_processed.csv",
    "program_profiles": DATA_DIR / "program_recommender_profiles_stage05.csv",
    "career": DATA_DIR / "karier_processed.csv",
    "skill": DATA_DIR / "skill_awal_processed.csv",
    "roadmap": DATA_DIR / "roadmap_belajar_processed.csv",
    "normalisasi": DATA_DIR / "normalisasi_bahasa_processed.csv",
}

missing_files = [str(path) for path in dataset_paths.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError("File dataset berikut belum ditemukan:\n" + "\n".join(missing_files))

intent_df = pd.read_csv(dataset_paths["intent"])
program_df = pd.read_csv(dataset_paths["program"])
program_profiles_df = pd.read_csv(dataset_paths["program_profiles"])
career_df = pd.read_csv(dataset_paths["career"])
skill_df = pd.read_csv(dataset_paths["skill"])
roadmap_df = pd.read_csv(dataset_paths["roadmap"])
normalisasi_df = pd.read_csv(dataset_paths["normalisasi"])

dataset_summary = pd.DataFrame({
    "dataset": list(dataset_paths.keys()),
    "path": [str(path) for path in dataset_paths.values()],
    "jumlah_baris": [
        len(intent_df), len(program_df), len(program_profiles_df),
        len(career_df), len(skill_df), len(roadmap_df), len(normalisasi_df)
    ],
    "jumlah_kolom": [
        intent_df.shape[1], program_df.shape[1], program_profiles_df.shape[1],
        career_df.shape[1], skill_df.shape[1], roadmap_df.shape[1], normalisasi_df.shape[1]
    ]
})

display(dataset_summary)

,dataset,path,jumlah_baris,jumlah_kolom
0,intent,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TU...,12,21
1,program,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TU...,5,22
2,program_profiles,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TU...,5,25
3,career,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TU...,5,15
4,skill,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TU...,7,15
5,roadmap,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TU...,6,15
6,normalisasi,D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TU...,18,10


### Interpretasi Awal Dataset

Jika seluruh dataset berhasil di-load, berarti artefak hasil tahap sebelumnya sudah tersedia dan siap diintegrasikan. Dataset yang paling penting pada tahap ini adalah:

- `intent_dataset_processed.csv` sebagai referensi label intent.
- `program_recommender_profiles_stage05.csv` sebagai basis rekomendasi program studi.
- `roadmap_belajar_processed.csv` sebagai basis roadmap belajar.
- `karier_processed.csv` dan `skill_awal_processed.csv` sebagai basis mapping prospek karier dan skill awal.
- `normalisasi_bahasa_processed.csv` sebagai kamus normalisasi input user.

In [4]:
# ============================================================
# 4. Load Model Artifact Tahap 04 dan Tahap 05
# ============================================================

artifact_paths = {
    "intent_model": MODEL_DIR / "intent_classifier_best_model.joblib",
    "intent_vectorizer": MODEL_DIR / "intent_tfidf_vectorizer.joblib",
    "intent_metadata": MODEL_DIR / "intent_classifier_metadata.json",
    "recommender_vectorizer": MODEL_DIR / "program_recommender_tfidf_vectorizer.joblib",
    "recommender_matrix": MODEL_DIR / "program_recommender_matrix.joblib",
    "recommender_config": MODEL_DIR / "program_recommender_config_stage05.json",
}

missing_artifacts = [str(path) for path in artifact_paths.values() if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError("Model artifact berikut belum ditemukan:\n" + "\n".join(missing_artifacts))

intent_model = joblib.load(artifact_paths["intent_model"])
intent_vectorizer = joblib.load(artifact_paths["intent_vectorizer"])

with open(artifact_paths["intent_metadata"], "r", encoding="utf-8") as f:
    intent_metadata = json.load(f)

recommender_vectorizer = joblib.load(artifact_paths["recommender_vectorizer"])
recommender_matrix = joblib.load(artifact_paths["recommender_matrix"])

with open(artifact_paths["recommender_config"], "r", encoding="utf-8") as f:
    recommender_config = json.load(f)

print("Model intent       :", type(intent_model).__name__)
print("Vectorizer intent  :", type(intent_vectorizer).__name__)
print("Vectorizer recsys  :", type(recommender_vectorizer).__name__)
print("Matrix recsys shape:", recommender_matrix.shape)
print("Best model stage04 :", intent_metadata.get("best_model_name"))
print("Method stage05     :", recommender_config.get("method"))

Model intent       : LinearSVC
Vectorizer intent  : TfidfVectorizer
Vectorizer recsys  : TfidfVectorizer
Matrix recsys shape: (5, 795)
Best model stage04 : LinearSVC
Method stage05     : Content-Based Filtering using TF-IDF, Cosine Similarity, and Rule-Based Structured Scoring


### Interpretasi Artifact

Cell di atas memastikan bahwa dua komponen utama chatbot sudah siap:

1. **Intent Classifier** dari Tahap 04 digunakan untuk memahami maksud pertanyaan user.
2. **Program Recommender** dari Tahap 05 digunakan untuk menghasilkan rekomendasi program studi berbasis kemiripan profil input user terhadap profil program studi.

Catatan penting: jika metadata menunjukkan bahwa model intent masih baseline dan dataset masih kecil, maka hasil prediksi perlu dibaca sebagai prototipe akademik, bukan sistem produksi final.

In [5]:
# ============================================================
# 5. Validasi Schema Minimum
# ============================================================

required_columns = {
    "intent_df": ["utterance", "intent_label", "utterance_preprocessed"],
    "program_profiles_df": ["program_id", "nama_program_studi", "deskripsi_singkat", "program_profile_text"],
    "career_df": ["karier_id", "nama_karier", "program_studi_relevan_id_list"],
    "skill_df": ["skill_id", "nama_skill", "program_studi_relevan_id_list"],
    "roadmap_df": ["program_id", "fase", "urutan_fase", "tujuan_fase", "materi_pokok"],
    "normalisasi_df": ["kata_input_clean", "kata_baku_clean"],
}

dataframes = {
    "intent_df": intent_df,
    "program_profiles_df": program_profiles_df,
    "career_df": career_df,
    "skill_df": skill_df,
    "roadmap_df": roadmap_df,
    "normalisasi_df": normalisasi_df,
}

schema_check = []

for df_name, cols in required_columns.items():
    df = dataframes[df_name]
    for col in cols:
        schema_check.append({
            "dataframe": df_name,
            "required_column": col,
            "status": "OK" if col in df.columns else "MISSING"
        })

schema_check_df = pd.DataFrame(schema_check)
display(schema_check_df)

if (schema_check_df["status"] == "MISSING").any():
    raise ValueError("Terdapat kolom wajib yang belum tersedia. Periksa schema dataset.")
else:
    print("Schema minimum sudah sesuai.")

,dataframe,required_column,status
0,intent_df,utterance,OK
1,intent_df,intent_label,OK
2,intent_df,utterance_preprocessed,OK
3,program_profiles_df,program_id,OK
4,program_profiles_df,nama_program_studi,OK
5,program_profiles_df,deskripsi_singkat,OK
6,program_profiles_df,program_profile_text,OK
7,career_df,karier_id,OK
8,career_df,nama_karier,OK
9,career_df,program_studi_relevan_id_list,OK


Schema minimum sudah sesuai.


In [6]:
# ============================================================
# 6. Fungsi Text Cleaning, Normalisasi, dan Preprocessing
# ============================================================

# Stopword sederhana agar notebook tidak bergantung pada library tambahan.
STOPWORDS = {
    "saya", "aku", "anda", "kamu", "dia", "mereka", "kami", "kita",
    "yang", "dan", "atau", "di", "ke", "dari", "untuk", "dengan", "pada",
    "adalah", "itu", "ini", "apa", "bagaimana", "kalau", "jika", "agar",
    "mau", "ingin", "bisa", "dapat", "harus", "dulu", "ya", "dong",
    "nih", "sih", "lah", "pun", "nya", "sebagai", "dalam", "bidang",
    "masuk", "ambil", "pilih", "cocoknya", "cocok", "pas"
}

# Kamus tambahan untuk membantu input non-formal dan istilah umum.
EXTRA_NORMALIZATION = {
    "gw": "saya",
    "gue": "saya",
    "aku": "saya",
    "ngoding": "pemrograman",
    "coding": "pemrograman",
    "prodi": "program studi",
    "jurusan": "program studi",
    "kuliah": "program studi",
    "programmer": "software engineer",
    "developer": "software engineer",
    "it": "informatika",
    "ti": "teknik informatika",
    "dkv": "desain komunikasi visual",
    "uiux": "ui ux",
    "seneng": "suka",
    "resep": "suka",
    "dadi": "menjadi",
    "janten": "menjadi",
    "pengin": "ingin",
    "hoyong": "ingin",
    "naon": "apa",
    "opo": "apa",
    "kumaha": "bagaimana",
}

def build_normalization_map(normalisasi_df: pd.DataFrame) -> dict:
    """Membangun kamus normalisasi dari dataset dan kamus tambahan."""
    norm_map = {}

    for _, row in normalisasi_df.iterrows():
        key = str(row.get("kata_input_clean", "")).strip().lower()
        value = str(row.get("kata_baku_clean", "")).strip().lower()
        if key and value and key != "nan" and value != "nan":
            norm_map[key] = value

    norm_map.update(EXTRA_NORMALIZATION)
    return norm_map

NORMALIZATION_MAP = build_normalization_map(normalisasi_df)

def clean_text(text: str) -> str:
    """Membersihkan teks dari karakter non-alfanumerik."""
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_text(text: str) -> str:
    """Melakukan normalisasi token berdasarkan kamus normalisasi."""
    cleaned = clean_text(text)
    tokens = cleaned.split()

    normalized_tokens = []
    for token in tokens:
        replacement = NORMALIZATION_MAP.get(token, token)
        normalized_tokens.extend(str(replacement).split())

    return " ".join(normalized_tokens)

def preprocess_text(text: str) -> str:
    """Pipeline preprocessing sederhana untuk input user."""
    normalized = normalize_text(text)
    tokens = normalized.split()
    filtered_tokens = [
        token for token in tokens
        if token not in STOPWORDS and len(token) > 1
    ]
    return " ".join(filtered_tokens)

# Contoh uji preprocessing
sample_texts = [
    "Aku seneng ngoding sama statistik, cocoknya kuliah apa ya?",
    "Aku pengin dadi analis data, jurusan sing pas apa?",
    "Abdi resep komputer jeung angka, prodi naon nu cocog?"
]

for text in sample_texts:
    print("Input       :", text)
    print("Normalized  :", normalize_text(text))
    print("Preprocessed:", preprocess_text(text))
    print("-" * 80)

Input       : Aku seneng ngoding sama statistik, cocoknya kuliah apa ya?
Normalized  : saya suka pemrograman sama statistik cocoknya program studi apa ya
Preprocessed: suka pemrograman sama statistik program studi
--------------------------------------------------------------------------------
Input       : Aku pengin dadi analis data, jurusan sing pas apa?
Normalized  : saya ingin menjadi analis data program studi yang pas apa
Preprocessed: menjadi analis data program studi
--------------------------------------------------------------------------------
Input       : Abdi resep komputer jeung angka, prodi naon nu cocog?
Normalized  : abdi suka komputer jeung angka program studi apa nu cocog
Preprocessed: abdi suka komputer jeung angka program studi nu cocog
--------------------------------------------------------------------------------


### Interpretasi Preprocessing

Fungsi preprocessing dibuat sederhana agar notebook tetap mudah dijalankan di Jupyter/Google Colab tanpa ketergantungan library tambahan. Normalisasi kata digunakan untuk membantu variasi input seperti bahasa non-formal, Jawa umum, dan Sunda umum. Output `preprocess_text()` akan menjadi input untuk model intent classification dan recommender system.

In [7]:
# ============================================================
# 7. Fungsi Intent Classification
# ============================================================

def softmax(values: np.ndarray) -> np.ndarray:
    """Mengubah skor decision_function menjadi pseudo-probability."""
    values = np.asarray(values, dtype=float)
    values = values - np.max(values)
    exp_values = np.exp(values)
    return exp_values / exp_values.sum()

def predict_intent(user_text: str) -> dict:
    """Memprediksi intent menggunakan model hasil Tahap 04."""
    processed_text = preprocess_text(user_text)
    vectorized_text = intent_vectorizer.transform([processed_text])

    predicted_label = str(intent_model.predict(vectorized_text)[0])
    labels = list(getattr(intent_model, "classes_", intent_metadata.get("labels", [])))

    confidence = None
    top_intents = []

    if hasattr(intent_model, "predict_proba"):
        probabilities = intent_model.predict_proba(vectorized_text)[0]
        confidence = float(np.max(probabilities))
        top_intents = sorted(
            [{"intent": str(label), "score": float(score)} for label, score in zip(labels, probabilities)],
            key=lambda x: x["score"],
            reverse=True
        )

    elif hasattr(intent_model, "decision_function"):
        decision_scores = intent_model.decision_function(vectorized_text)

        if np.asarray(decision_scores).ndim == 1:
            decision_scores = np.asarray(decision_scores).reshape(1, -1)

        pseudo_probs = softmax(decision_scores[0])
        confidence = float(np.max(pseudo_probs))

        top_intents = sorted(
            [{"intent": str(label), "score": float(score)} for label, score in zip(labels, pseudo_probs)],
            key=lambda x: x["score"],
            reverse=True
        )

    else:
        confidence = 1.0
        top_intents = [{"intent": predicted_label, "score": 1.0}]

    return {
        "raw_text": user_text,
        "processed_text": processed_text,
        "predicted_intent": predicted_label,
        "confidence": round(confidence, 4),
        "top_intents": top_intents[:5]
    }

# Uji cepat intent classification
intent_test = "Saya suka matematika dan data, program studi apa yang cocok?"
intent_result = predict_intent(intent_test)
intent_result

{'raw_text': 'Saya suka matematika dan data, program studi apa yang cocok?',
 'processed_text': 'suka matematika data program studi',
 'predicted_intent': 'rekomendasi_prodi',
 'confidence': 0.2259,
 'top_intents': [{'intent': 'rekomendasi_prodi', 'score': 0.2259326156357707},
  {'intent': 'sapaan', 'score': 0.12785653488228144},
  {'intent': 'fallback', 'score': 0.12060478584062263},
  {'intent': 'skill_awal', 'score': 0.11220863886002},
  {'intent': 'roadmap_belajar', 'score': 0.10937030406972577}]}

### Interpretasi Intent Classification

Fungsi `predict_intent()` menghasilkan tiga informasi penting:

- `processed_text`: teks setelah preprocessing.
- `predicted_intent`: hasil prediksi model.
- `confidence`: estimasi tingkat keyakinan model.

Karena model Tahap 04 menggunakan `LinearSVC`, nilai confidence dihitung menggunakan pendekatan pseudo-probability dari `decision_function`. Nilai ini bukan probabilitas terkalibrasi, tetapi cukup membantu untuk routing awal pada prototipe chatbot.

In [8]:
# ============================================================
# 8. Rule-Based Intent Routing
# ============================================================

INTENT_RULES = {
    "sapaan": [
        "halo", "hai", "hello", "assalamualaikum", "selamat pagi",
        "selamat siang", "selamat sore", "selamat malam"
    ],
    "roadmap_belajar": [
        "roadmap", "alur belajar", "belajar apa", "harus belajar",
        "mulai dari mana", "persiapan", "materi awal", "tahapan belajar"
    ],
    "prospek_karier": [
        "prospek kerja", "karier", "pekerjaan", "kerja apa",
        "peluang kerja", "profesi", "job", "masa depan"
    ],
    "skill_awal": [
        "skill", "kemampuan", "kompetensi", "belajar dulu",
        "materi dasar", "dasar apa", "tools"
    ],
    "info_program_studi": [
        "apa itu", "jelaskan", "informasi", "info",
        "mempelajari apa", "belajar apa di"
    ],
    "rekomendasi_prodi": [
        "rekomendasi", "program studi", "prodi", "jurusan",
        "cocok", "pilih", "kuliah apa", "suka", "minat"
    ],
    "klarifikasi_minat": [
        "bingung", "belum tahu", "belum tau", "tidak tahu",
        "ga tau", "gak tau", "masih ragu"
    ],
}

ROUTING_PRIORITY = [
    "sapaan",
    "roadmap_belajar",
    "prospek_karier",
    "skill_awal",
    "info_program_studi",
    "rekomendasi_prodi",
    "klarifikasi_minat",
]

def rule_based_intent(user_text: str) -> dict:
    """Mendeteksi intent berbasis keyword eksplisit."""
    raw = clean_text(user_text)
    normalized = normalize_text(user_text)
    processed = preprocess_text(user_text)

    combined_text = f"{raw} {normalized} {processed}"

    for intent in ROUTING_PRIORITY:
        matched_keywords = []
        for keyword in INTENT_RULES[intent]:
            keyword_clean = clean_text(keyword)
            if keyword_clean in combined_text:
                matched_keywords.append(keyword)

        if matched_keywords:
            score = min(1.0, 0.4 + 0.2 * len(matched_keywords))
            return {
                "rule_intent": intent,
                "rule_confidence": round(score, 4),
                "matched_keywords": matched_keywords
            }

    return {
        "rule_intent": None,
        "rule_confidence": 0.0,
        "matched_keywords": []
    }

def route_intent(user_text: str, model_threshold: float = 0.30) -> dict:
    """Menggabungkan model intent dan rule-based intent menjadi final intent."""
    model_result = predict_intent(user_text)
    rule_result = rule_based_intent(user_text)

    model_intent = model_result["predicted_intent"]
    model_confidence = model_result["confidence"]
    rule_intent = rule_result["rule_intent"]
    rule_confidence = rule_result["rule_confidence"]

    if rule_intent is not None and rule_confidence >= 0.40:
        final_intent = rule_intent
        source = "rule_based"
    elif model_confidence >= model_threshold:
        final_intent = model_intent
        source = "model"
    else:
        final_intent = "fallback"
        source = "fallback_threshold"

    return {
        "final_intent": final_intent,
        "routing_source": source,
        "model_result": model_result,
        "rule_result": rule_result
    }

# Uji intent routing
routing_examples = [
    "Halo, saya mau tanya jurusan kuliah.",
    "Saya suka matematika dan data, program studi apa yang cocok?",
    "Kalau ingin masuk Sains Data, saya harus belajar apa dulu?",
    "Prospek kerja Sistem Informasi apa saja?",
    "Saya ingin beli sepatu olahraga."
]

for text in routing_examples:
    print("Input:", text)
    print(route_intent(text))
    print("-" * 100)

Input: Halo, saya mau tanya jurusan kuliah.
{'final_intent': 'sapaan', 'routing_source': 'rule_based', 'model_result': {'raw_text': 'Halo, saya mau tanya jurusan kuliah.', 'processed_text': 'halo tanya program studi program studi', 'predicted_intent': 'sapaan', 'confidence': 0.2121, 'top_intents': [{'intent': 'sapaan', 'score': 0.2120543393052468}, {'intent': 'fallback', 'score': 0.15135585746292995}, {'intent': 'rekomendasi_prodi', 'score': 0.130900226643856}, {'intent': 'klarifikasi_minat', 'score': 0.10284086646384318}, {'intent': 'info_program_studi', 'score': 0.10233963386802086}]}, 'rule_result': {'rule_intent': 'sapaan', 'rule_confidence': 0.6, 'matched_keywords': ['halo']}}
----------------------------------------------------------------------------------------------------
Input: Saya suka matematika dan data, program studi apa yang cocok?
{'final_intent': 'rekomendasi_prodi', 'routing_source': 'rule_based', 'model_result': {'raw_text': 'Saya suka matematika dan data, program s

### Interpretasi Intent Routing

Routing dibuat secara hybrid:

- **Model-based routing** menggunakan model intent classification.
- **Rule-based routing** digunakan sebagai pengaman untuk intent yang memiliki kata kunci sangat eksplisit, misalnya `roadmap`, `prospek kerja`, `skill`, atau `apa itu`.

Pendekatan hybrid ini realistis untuk prototipe akademik karena dataset intent saat ini masih kecil. Rule-based routing membantu menjaga respons chatbot agar tetap relevan ketika model belum cukup kuat.

In [9]:
# ============================================================
# 9. Helper Function untuk Mapping Program, Karier, Skill, dan Roadmap
# ============================================================

def split_values(value) -> list:
    """Memecah nilai string dengan pemisah pipe atau koma menjadi list."""
    if pd.isna(value):
        return []
    value = str(value).strip()
    if not value:
        return []
    return [item.strip() for item in re.split(r"\||,", value) if item.strip()]

def get_career_names_from_program(program_row: pd.Series) -> list:
    """Mengambil nama karier dari kolom mapped atau dari ID karier."""
    if "mapped_career_names" in program_row.index and pd.notna(program_row["mapped_career_names"]):
        return split_values(str(program_row["mapped_career_names"]).replace(", ", "|"))

    career_ids = split_values(program_row.get("prospek_karier_id_list", ""))
    names = career_df[career_df["karier_id"].isin(career_ids)]["nama_karier"].tolist()
    return names

def get_skill_names_from_program(program_row: pd.Series) -> list:
    """Mengambil nama skill dari kolom mapped atau dari ID skill."""
    if "mapped_skill_names" in program_row.index and pd.notna(program_row["mapped_skill_names"]):
        return split_values(str(program_row["mapped_skill_names"]).replace(", ", "|"))

    skill_ids = split_values(program_row.get("skill_id_list", ""))
    names = skill_df[skill_df["skill_id"].isin(skill_ids)]["nama_skill"].tolist()
    return names

def get_roadmap_by_program_id(program_id: str) -> pd.DataFrame:
    """Mengambil roadmap berdasarkan program_id."""
    roadmap = roadmap_df[roadmap_df["program_id"] == program_id].copy()

    if "urutan_fase" in roadmap.columns:
        roadmap = roadmap.sort_values("urutan_fase")

    return roadmap.reset_index(drop=True)

def build_structured_profile_text(row: pd.Series) -> str:
    """Menggabungkan kolom profil terstruktur untuk structured matching."""
    candidate_columns = [
        "nama_program_studi",
        "rumpun_ilmu",
        "bidang_keilmuan",
        "deskripsi_singkat",
        "minat_cocok",
        "mapel_relevan",
        "hobi_relevan",
        "gaya_belajar_cocok",
        "karakteristik_cocok",
        "keyword_rekomendasi",
        "mapped_career_text",
        "mapped_career_names",
        "mapped_skill_names"
    ]

    values = []
    for col in candidate_columns:
        if col in row.index and pd.notna(row[col]):
            values.append(str(row[col]))

    return " ".join(values)

def keyword_overlap(query_text: str, profile_text: str) -> tuple:
    """Menghitung overlap keyword antara input user dan profil program."""
    query_tokens = set(preprocess_text(query_text).split())
    profile_tokens = set(preprocess_text(profile_text).split())

    if not query_tokens:
        return 0.0, []

    matched_tokens = sorted(list(query_tokens.intersection(profile_tokens)))
    score = len(matched_tokens) / max(1, min(len(query_tokens), 10))
    return min(score, 1.0), matched_tokens

print("Helper function berhasil dibuat.")

Helper function berhasil dibuat.


In [10]:
# ============================================================
# 10. Fungsi Program Studi Recommender
# ============================================================

TEXT_SIMILARITY_WEIGHT = float(recommender_config.get("text_similarity_weight", 0.65))
STRUCTURED_SCORE_WEIGHT = float(recommender_config.get("structured_score_weight", 0.35))

def recommend_programs(user_text: str, top_n: int = 3) -> pd.DataFrame:
    """Memberikan rekomendasi program studi berdasarkan input user."""
    processed_query = preprocess_text(user_text)

    # Jika hasil preprocessing kosong, gunakan normalized text agar vectorizer tetap menerima input.
    if not processed_query:
        processed_query = normalize_text(user_text)

    query_vector = recommender_vectorizer.transform([processed_query])
    text_similarities = cosine_similarity(query_vector, recommender_matrix).ravel()

    recommendations = []

    for idx, row in program_profiles_df.iterrows():
        structured_text = build_structured_profile_text(row)
        structured_score, matched_tokens = keyword_overlap(user_text, structured_text)

        text_similarity = float(text_similarities[idx])
        final_score = (
            TEXT_SIMILARITY_WEIGHT * text_similarity
            + STRUCTURED_SCORE_WEIGHT * structured_score
        )

        career_names = get_career_names_from_program(row)
        skill_names = get_skill_names_from_program(row)

        recommendations.append({
            "rank": None,
            "program_id": row.get("program_id"),
            "nama_program_studi": row.get("nama_program_studi"),
            "jenjang": row.get("jenjang"),
            "rumpun_ilmu": row.get("rumpun_ilmu"),
            "bidang_keilmuan": row.get("bidang_keilmuan"),
            "deskripsi_singkat": row.get("deskripsi_singkat"),
            "text_similarity": round(text_similarity, 4),
            "structured_score": round(structured_score, 4),
            "final_score": round(final_score, 4),
            "matched_keywords": ", ".join(matched_tokens[:12]),
            "prospek_karier": ", ".join(career_names),
            "skill_awal": ", ".join(skill_names),
        })

    rec_df = pd.DataFrame(recommendations)
    rec_df = rec_df.sort_values("final_score", ascending=False).head(top_n).reset_index(drop=True)
    rec_df["rank"] = rec_df.index + 1

    return rec_df

# Uji recommender
recommend_programs("Saya suka data, matematika, statistik, Python, dan machine learning.", top_n=3)

,rank,program_id,nama_program_studi,jenjang,rumpun_ilmu,bidang_keilmuan,deskripsi_singkat,text_similarity,structured_score,final_score,matched_keywords,prospek_karier,skill_awal
0,1,PRG003,Sains Data,S1,Komputer,Data Science dan Analitik,"Mempelajari statistika, pemrograman, machine l...",0.4743,1.0000,0.6583,"data, learning, machine, matematika, python, s...","Data Analyst, Data Scientist","Dasar Pemrograman Python, Dasar Basis Data dan..."
1,2,PRG004,Statistika,S1,Matematika dan Sains,Statistika Terapan,"Mempelajari pengumpulan data, probabilitas, pe...",0.3520,0.8571,0.5288,"data, learning, machine, matematika, python, s...","Data Analyst, Data Scientist","Analisis Data Dasar, Statistika Dasar"
2,3,PRG001,Teknik Informatika,S1,Komputer,Software Engineering dan Komputasi,"Mempelajari pemrograman, algoritma, basis data...",0.0615,0.5714,0.2400,"data, matematika, python, suka",Software Engineer,"Dasar Pemrograman Python, Dasar Basis Data dan..."


### Interpretasi Recommender

Fungsi `recommend_programs()` menggunakan gabungan dua pendekatan:

1. **Text Similarity**: menghitung kemiripan input user dengan profil program studi menggunakan TF-IDF dan cosine similarity.
2. **Structured Score**: menghitung kecocokan keyword input user dengan atribut terstruktur seperti minat, mapel relevan, hobi, gaya belajar, prospek karier, dan skill.

Skor akhir mengikuti konfigurasi Tahap 05, yaitu kombinasi text similarity dan structured score.

In [11]:
# ============================================================
# 11. Deteksi Program Studi dari Input User
# ============================================================

def find_program_from_text(user_text: str) -> pd.Series | None:
    """Mendeteksi program studi eksplisit dari input user."""
    raw_clean = clean_text(user_text)
    processed = preprocess_text(user_text)

    # 1. Cek nama program studi secara langsung.
    for _, row in program_profiles_df.iterrows():
        program_name = clean_text(row.get("nama_program_studi", ""))
        if program_name and program_name in raw_clean:
            return row

    # 2. Cek keyword rekomendasi.
    best_row = None
    best_score = 0.0

    for _, row in program_profiles_df.iterrows():
        keyword_text = " ".join([
            str(row.get("nama_program_studi", "")),
            str(row.get("keyword_rekomendasi", "")),
            str(row.get("bidang_keilmuan", "")),
            str(row.get("program_profile_text", "")),
        ])

        score, _ = keyword_overlap(processed, keyword_text)
        if score > best_score:
            best_score = score
            best_row = row

    if best_score >= 0.20:
        return best_row

    # 3. Jika tidak ditemukan eksplisit, gunakan top-1 recommender.
    top_rec = recommend_programs(user_text, top_n=1)

    if not top_rec.empty and float(top_rec.iloc[0]["final_score"]) > 0:
        program_id = top_rec.iloc[0]["program_id"]
        matched = program_profiles_df[program_profiles_df["program_id"] == program_id]
        if not matched.empty:
            return matched.iloc[0]

    return None

# Uji deteksi program studi
for text in [
    "Apa itu Teknik Informatika?",
    "Kalau saya mau masuk Sains Data harus belajar apa?",
    "Saya suka desain visual dan membuat konten."
]:
    program = find_program_from_text(text)
    print("Input:", text)
    print("Detected:", None if program is None else program["nama_program_studi"])
    print("-" * 80)

Input: Apa itu Teknik Informatika?
Detected: Teknik Informatika
--------------------------------------------------------------------------------
Input: Kalau saya mau masuk Sains Data harus belajar apa?
Detected: Sains Data
--------------------------------------------------------------------------------
Input: Saya suka desain visual dan membuat konten.
Detected: Desain Komunikasi Visual
--------------------------------------------------------------------------------


In [12]:
# ============================================================
# 12. Formatter Respons Chatbot
# ============================================================

def format_recommendation_response(user_text: str, top_n: int = 3) -> tuple[str, pd.DataFrame]:
    rec_df = recommend_programs(user_text, top_n=top_n)

    if rec_df.empty:
        return (
            "Saya belum dapat memberikan rekomendasi program studi. "
            "Coba jelaskan minat, mata pelajaran favorit, hobi, gaya belajar, atau tujuan karier Anda.",
            rec_df
        )

    lines = []
    lines.append("Berikut rekomendasi awal program studi berdasarkan profil yang Anda sampaikan:\n")

    for _, row in rec_df.iterrows():
        lines.append(f"{int(row['rank'])}. {row['nama_program_studi']} ({row['jenjang']})")
        lines.append(f"   - Bidang: {row['bidang_keilmuan']}")
        lines.append(f"   - Alasan: cocok dengan keyword `{row['matched_keywords']}`.")
        lines.append(f"   - Skor akhir: {row['final_score']}")
        lines.append(f"   - Prospek karier: {row['prospek_karier']}")
        lines.append(f"   - Skill awal: {row['skill_awal']}")
        lines.append("")

    lines.append(
        "Catatan: rekomendasi ini bersifat awal dan sebaiknya divalidasi kembali dengan minat, nilai akademik, "
        "portofolio, serta konsultasi dengan guru BK atau pembimbing akademik."
    )

    return "\n".join(lines), rec_df

def format_roadmap_response(user_text: str) -> tuple[str, pd.DataFrame]:
    program = find_program_from_text(user_text)

    if program is None:
        return (
            "Saya belum dapat menentukan program studi yang dimaksud. "
            "Mohon sebutkan program studi atau minat Anda, misalnya Sains Data, Teknik Informatika, atau Sistem Informasi.",
            pd.DataFrame()
        )

    roadmap = get_roadmap_by_program_id(program["program_id"])

    if roadmap.empty:
        return (
            f"Roadmap untuk {program['nama_program_studi']} belum tersedia pada dataset saat ini.",
            roadmap
        )

    lines = []
    lines.append(f"Roadmap belajar awal untuk {program['nama_program_studi']} ({program['jenjang']}):\n")

    for _, row in roadmap.iterrows():
        lines.append(f"Fase {row['urutan_fase']} — {row['fase']} ({row['durasi_rekomendasi']})")
        lines.append(f"- Tujuan: {row['tujuan_fase']}")
        lines.append(f"- Materi pokok: {row['materi_pokok']}")
        lines.append(f"- Aktivitas praktik: {row['aktivitas_praktik']}")
        lines.append(f"- Output portofolio: {row['output_portofolio']}")
        lines.append(f"- Tools: {row['tools_umum']}")
        lines.append("")

    return "\n".join(lines), roadmap

def format_career_response(user_text: str) -> tuple[str, pd.DataFrame]:
    program = find_program_from_text(user_text)

    if program is None:
        return (
            "Saya belum dapat menentukan program studi yang dimaksud untuk menampilkan prospek karier.",
            pd.DataFrame()
        )

    career_names = get_career_names_from_program(program)

    lines = []
    lines.append(f"Prospek karier yang relevan untuk {program['nama_program_studi']}:\n")

    if not career_names:
        lines.append("- Data prospek karier belum tersedia.")
    else:
        for career_name in career_names:
            matched = career_df[career_df["nama_karier"] == career_name]
            if matched.empty:
                lines.append(f"- {career_name}")
            else:
                career = matched.iloc[0]
                lines.append(f"- {career['nama_karier']} ({career['bidang_karier']})")
                lines.append(f"  Tugas utama: {career['tugas_utama']}")
                lines.append(f"  Skill teknis: {career['skill_teknis']}")

    return "\n".join(lines), pd.DataFrame({"prospek_karier": career_names})

def format_skill_response(user_text: str) -> tuple[str, pd.DataFrame]:
    program = find_program_from_text(user_text)

    if program is None:
        return (
            "Saya belum dapat menentukan program studi yang dimaksud untuk menampilkan skill awal.",
            pd.DataFrame()
        )

    skill_names = get_skill_names_from_program(program)

    lines = []
    lines.append(f"Skill awal yang disarankan untuk {program['nama_program_studi']}:\n")

    if not skill_names:
        lines.append("- Data skill awal belum tersedia.")
    else:
        for skill_name in skill_names:
            matched = skill_df[skill_df["nama_skill"] == skill_name]
            if matched.empty:
                lines.append(f"- {skill_name}")
            else:
                skill = matched.iloc[0]
                lines.append(f"- {skill['nama_skill']}")
                lines.append(f"  Deskripsi: {skill['deskripsi_singkat']}")
                lines.append(f"  Estimasi belajar: {skill['estimasi_waktu_belajar']}")
                lines.append(f"  Sumber belajar awal: {skill['sumber_belajar_awal']}")

    return "\n".join(lines), pd.DataFrame({"skill_awal": skill_names})

def format_program_info_response(user_text: str) -> tuple[str, pd.DataFrame]:
    program = find_program_from_text(user_text)

    if program is None:
        return (
            "Saya belum dapat menemukan program studi yang dimaksud. "
            "Coba sebutkan nama program studi, misalnya Teknik Informatika, Sistem Informasi, Sains Data, Statistika, atau DKV.",
            pd.DataFrame()
        )

    career_names = get_career_names_from_program(program)
    skill_names = get_skill_names_from_program(program)

    lines = []
    lines.append(f"Informasi singkat tentang {program['nama_program_studi']} ({program['jenjang']}):\n")
    lines.append(f"- Rumpun ilmu: {program['rumpun_ilmu']}")
    lines.append(f"- Bidang keilmuan: {program['bidang_keilmuan']}")
    lines.append(f"- Deskripsi: {program['deskripsi_singkat']}")
    lines.append(f"- Minat yang cocok: {program.get('minat_cocok', '-')}")
    lines.append(f"- Mata pelajaran relevan: {program.get('mapel_relevan', '-')}")
    lines.append(f"- Prospek karier: {', '.join(career_names)}")
    lines.append(f"- Skill awal: {', '.join(skill_names)}")

    return "\n".join(lines), pd.DataFrame([program.to_dict()])

def format_clarification_response() -> str:
    return (
        "Agar saya bisa memberikan rekomendasi yang lebih tepat, coba jawab beberapa poin berikut:\n"
        "1. Mata pelajaran apa yang paling Anda sukai?\n"
        "2. Aktivitas atau hobi apa yang paling sering Anda lakukan?\n"
        "3. Apakah Anda lebih suka belajar dengan praktik, membaca teori, berdiskusi, atau membuat karya visual?\n"
        "4. Apakah Anda punya gambaran karier yang diinginkan?"
    )

print("Formatter respons chatbot berhasil dibuat.")

Formatter respons chatbot berhasil dibuat.


In [13]:
# ============================================================
# 13. Fungsi Utama chatbot_response()
# ============================================================

def chatbot_response(user_input: str, top_n: int = 3, return_dict: bool = False):
    """Fungsi utama chatbot EduPath Career Assistant."""
    routing = route_intent(user_input)
    final_intent = routing["final_intent"]

    payload = None

    if final_intent == "sapaan":
        response_text = (
            "Halo, saya EduPath Career Assistant. "
            "Saya bisa membantu memberikan rekomendasi program studi S1, roadmap belajar awal, "
            "prospek karier, dan skill awal berdasarkan minat Anda. "
            "Silakan ceritakan minat, mapel favorit, hobi, gaya belajar, atau tujuan karier Anda."
        )

    elif final_intent == "rekomendasi_prodi":
        response_text, payload = format_recommendation_response(user_input, top_n=top_n)

    elif final_intent == "roadmap_belajar":
        response_text, payload = format_roadmap_response(user_input)

    elif final_intent == "prospek_karier":
        response_text, payload = format_career_response(user_input)

    elif final_intent == "skill_awal":
        response_text, payload = format_skill_response(user_input)

    elif final_intent == "info_program_studi":
        response_text, payload = format_program_info_response(user_input)

    elif final_intent == "klarifikasi_minat":
        response_text = format_clarification_response()

    else:
        response_text = (
            "Maaf, saya belum memahami pertanyaan Anda. "
            "Coba jelaskan minat, mata pelajaran favorit, hobi, gaya belajar, atau tujuan karier Anda. "
            "Contoh: 'Saya suka matematika dan data, program studi apa yang cocok?'"
        )

    result = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "user_input": user_input,
        "final_intent": final_intent,
        "routing_source": routing["routing_source"],
        "model_predicted_intent": routing["model_result"]["predicted_intent"],
        "model_confidence": routing["model_result"]["confidence"],
        "rule_intent": routing["rule_result"]["rule_intent"],
        "rule_confidence": routing["rule_result"]["rule_confidence"],
        "matched_rule_keywords": ", ".join(routing["rule_result"]["matched_keywords"]),
        "processed_text": routing["model_result"]["processed_text"],
        "response_text": response_text,
        "payload": payload
    }

    if return_dict:
        return result

    return response_text

# Uji fungsi utama chatbot_response()
print(chatbot_response("Saya suka data, matematika, statistik, dan ingin jadi data scientist. Program studi apa yang cocok?"))

Berikut rekomendasi awal program studi berdasarkan profil yang Anda sampaikan:

1. Sains Data (S1)
   - Bidang: Data Science dan Analitik
   - Alasan: cocok dengan keyword `data, matematika, scientist, statistik, suka`.
   - Skor akhir: 0.4186
   - Prospek karier: Data Analyst, Data Scientist
   - Skill awal: Dasar Pemrograman Python, Dasar Basis Data dan SQL, Analisis Data Dasar, Statistika Dasar

2. Statistika (S1)
   - Bidang: Statistika Terapan
   - Alasan: cocok dengan keyword `data, matematika, scientist, statistik`.
   - Skor akhir: 0.3615
   - Prospek karier: Data Analyst, Data Scientist
   - Skill awal: Analisis Data Dasar, Statistika Dasar

3. Sistem Informasi (S1)
   - Bidang: Sistem Bisnis dan Teknologi Informasi
   - Alasan: cocok dengan keyword `data, jadi, matematika`.
   - Skor akhir: 0.1967
   - Prospek karier: Business Analyst, Data Analyst
   - Skill awal: Dasar Basis Data dan SQL, Analisis Data Dasar, Komunikasi dan Analisis Kebutuhan

Catatan: rekomendasi ini bersi

### Interpretasi Fungsi `chatbot_response()`

Fungsi `chatbot_response()` merupakan inti integrasi Tahap 06. Fungsi ini menggabungkan:

- preprocessing input,
- prediksi intent,
- rule-based routing,
- recommender system,
- mapping roadmap,
- mapping prospek karier,
- mapping skill awal,
- dan response generation.

Dengan struktur ini, chatbot sudah dapat diuji menggunakan beberapa skenario input secara end-to-end.

In [14]:
# ============================================================
# 14. Pengujian Beberapa Skenario Input
# ============================================================

test_scenarios = [
    {
        "scenario": "Sapaan",
        "input": "Halo kak, saya mau tanya jurusan kuliah."
    },
    {
        "scenario": "Rekomendasi prodi formal",
        "input": "Saya suka matematika, statistik, dan data. Program studi apa yang cocok?"
    },
    {
        "scenario": "Rekomendasi prodi non-formal",
        "input": "Aku seneng ngoding dan bikin aplikasi, cocok kuliah apa ya?"
    },
    {
        "scenario": "Bahasa Jawa umum",
        "input": "Aku pengin dadi analis data, jurusan sing pas apa?"
    },
    {
        "scenario": "Bahasa Sunda umum",
        "input": "Abdi resep komputer jeung angka, prodi naon nu cocog?"
    },
    {
        "scenario": "Roadmap belajar",
        "input": "Kalau saya mau masuk Sains Data, harus belajar apa dulu?"
    },
    {
        "scenario": "Prospek karier",
        "input": "Prospek kerja Sistem Informasi itu apa saja?"
    },
    {
        "scenario": "Skill awal",
        "input": "Skill awal untuk menjadi UI UX Designer apa saja?"
    },
    {
        "scenario": "Info program studi",
        "input": "Apa itu Teknik Informatika?"
    },
    {
        "scenario": "Fallback",
        "input": "Saya ingin beli sepatu olahraga."
    },
]

test_results = []

for case in test_scenarios:
    result = chatbot_response(case["input"], return_dict=True)

    test_results.append({
        "scenario": case["scenario"],
        "user_input": case["input"],
        "final_intent": result["final_intent"],
        "routing_source": result["routing_source"],
        "model_predicted_intent": result["model_predicted_intent"],
        "model_confidence": result["model_confidence"],
        "rule_intent": result["rule_intent"],
        "rule_confidence": result["rule_confidence"],
        "matched_rule_keywords": result["matched_rule_keywords"],
        "processed_text": result["processed_text"],
        "response_text": result["response_text"],
    })

    print("=" * 110)
    print("SCENARIO:", case["scenario"])
    print("INPUT   :", case["input"])
    print("INTENT  :", result["final_intent"], "| source:", result["routing_source"])
    print("-" * 110)
    print(result["response_text"])
    print()

test_results_df = pd.DataFrame(test_results)
display(test_results_df[[
    "scenario", "final_intent", "routing_source",
    "model_predicted_intent", "model_confidence",
    "rule_intent", "rule_confidence"
]])

SCENARIO: Sapaan
INPUT   : Halo kak, saya mau tanya jurusan kuliah.
INTENT  : sapaan | source: rule_based
--------------------------------------------------------------------------------------------------------------
Halo, saya EduPath Career Assistant. Saya bisa membantu memberikan rekomendasi program studi S1, roadmap belajar awal, prospek karier, dan skill awal berdasarkan minat Anda. Silakan ceritakan minat, mapel favorit, hobi, gaya belajar, atau tujuan karier Anda.

SCENARIO: Rekomendasi prodi formal
INPUT   : Saya suka matematika, statistik, dan data. Program studi apa yang cocok?
INTENT  : rekomendasi_prodi | source: rule_based
--------------------------------------------------------------------------------------------------------------
Berikut rekomendasi awal program studi berdasarkan profil yang Anda sampaikan:

1. Sains Data (S1)
   - Bidang: Data Science dan Analitik
   - Alasan: cocok dengan keyword `data, matematika, statistik, suka`.
   - Skor akhir: 0.3194
   - Prospek

,scenario,final_intent,routing_source,model_predicted_intent,model_confidence,rule_intent,rule_confidence
0,Sapaan,sapaan,rule_based,sapaan,0.2121,sapaan,0.6
1,Rekomendasi prodi formal,rekomendasi_prodi,rule_based,rekomendasi_prodi,0.2424,rekomendasi_prodi,1.0
2,Rekomendasi prodi non-formal,rekomendasi_prodi,rule_based,rekomendasi_prodi,0.2287,rekomendasi_prodi,1.0
3,Bahasa Jawa umum,rekomendasi_prodi,rule_based,rekomendasi_prodi,0.1643,rekomendasi_prodi,0.8
4,Bahasa Sunda umum,rekomendasi_prodi,rule_based,rekomendasi_prodi,0.2565,rekomendasi_prodi,1.0
5,Roadmap belajar,roadmap_belajar,rule_based,roadmap_belajar,0.2046,roadmap_belajar,0.8
6,Prospek karier,prospek_karier,rule_based,prospek_karier,0.2087,prospek_karier,0.6
7,Skill awal,skill_awal,rule_based,skill_awal,0.2718,skill_awal,0.6
8,Info program studi,info_program_studi,rule_based,info_program_studi,0.2579,info_program_studi,0.8
9,Fallback,fallback,fallback_threshold,rekomendasi_prodi,0.1477,None,0.0


### Interpretasi Hasil Pengujian Skenario

Hasil uji skenario digunakan untuk memastikan bahwa pipeline chatbot mampu merespons beberapa jenis pertanyaan:

- Sapaan awal.
- Rekomendasi program studi berbasis minat.
- Input Bahasa Indonesia non-formal.
- Input Jawa umum dan Sunda umum.
- Permintaan roadmap belajar.
- Permintaan prospek karier.
- Permintaan skill awal.
- Permintaan informasi program studi.
- Input tidak relevan sebagai fallback.

Jika terdapat skenario yang belum tepat, penyebab yang paling mungkin adalah ukuran dataset intent yang masih kecil, variasi bahasa yang belum luas, atau keyword routing yang perlu ditambah.

In [15]:
# ============================================================
# 15. Penyimpanan Output Report Tahap 06
# ============================================================

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

test_results_path = REPORT_DIR / "chatbot_test_results_stage06.csv"
sample_responses_path = REPORT_DIR / "chatbot_sample_responses_stage06.md"
summary_path = REPORT_DIR / "stage06_summary.txt"

# Simpan hasil uji skenario ke CSV
test_results_df.to_csv(test_results_path, index=False, encoding="utf-8-sig")

# Simpan contoh respons chatbot ke Markdown
md_lines = []
md_lines.append("# Chatbot Sample Responses — Stage 06")
md_lines.append("")
md_lines.append(f"Generated at: {timestamp}")
md_lines.append("")

for _, row in test_results_df.iterrows():
    md_lines.append(f"## {row['scenario']}")
    md_lines.append("")
    md_lines.append(f"**Input:** {row['user_input']}")
    md_lines.append("")
    md_lines.append(f"**Final Intent:** `{row['final_intent']}`")
    md_lines.append("")
    md_lines.append("**Response:**")
    md_lines.append("")
    md_lines.append(str(row["response_text"]))
    md_lines.append("")

sample_responses_path.write_text("\n".join(md_lines), encoding="utf-8")

# Simpan summary tahap 06
summary_text = f"""
TAHAP 06 — Chatbot Pipeline Integration

Project:
EduPath Career Assistant

Generated at:
{timestamp}

Tujuan:
Mengintegrasikan model intent classification dari Tahap 04 dan program studi recommender system
dari Tahap 05 ke dalam satu pipeline chatbot sederhana.

Komponen Pipeline:
1. Text cleaning
2. Normalisasi bahasa
3. Text preprocessing
4. Intent classification
5. Rule-based intent routing
6. Program studi recommender
7. Roadmap mapping
8. Career mapping
9. Skill mapping
10. Chatbot response generation

Artifact yang digunakan:
1. {artifact_paths["intent_model"].relative_to(PROJECT_ROOT)}
2. {artifact_paths["intent_vectorizer"].relative_to(PROJECT_ROOT)}
3. {artifact_paths["recommender_vectorizer"].relative_to(PROJECT_ROOT)}
4. {artifact_paths["recommender_matrix"].relative_to(PROJECT_ROOT)}

Dataset utama:
1. {dataset_paths["intent"].relative_to(PROJECT_ROOT)}
2. {dataset_paths["program_profiles"].relative_to(PROJECT_ROOT)}
3. {dataset_paths["career"].relative_to(PROJECT_ROOT)}
4. {dataset_paths["skill"].relative_to(PROJECT_ROOT)}
5. {dataset_paths["roadmap"].relative_to(PROJECT_ROOT)}
6. {dataset_paths["normalisasi"].relative_to(PROJECT_ROOT)}

Jumlah skenario uji:
{len(test_results_df)}

Catatan Evaluasi:
Pipeline chatbot sudah berjalan end-to-end untuk intent routing, rekomendasi program studi,
roadmap belajar, prospek karier, skill awal, info program studi, sapaan, dan fallback.
Namun, karena dataset intent dan program studi masih kecil, hasil rekomendasi dan klasifikasi
masih bersifat baseline akademik dan perlu diperkuat dengan perluasan dataset serta validasi domain expert.

Output Stage 06:
1. reports/stage06/chatbot_test_results_stage06.csv
2. reports/stage06/chatbot_sample_responses_stage06.md
3. reports/stage06/stage06_summary.txt
""".strip()

summary_path.write_text(summary_text, encoding="utf-8")

print("Report berhasil disimpan:")
print("-", test_results_path)
print("-", sample_responses_path)
print("-", summary_path)

Report berhasil disimpan:
- D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage06\chatbot_test_results_stage06.csv
- D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage06\chatbot_sample_responses_stage06.md
- D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage06\stage06_summary.txt


In [16]:
# ============================================================
# 16. Checklist File Hasil Tahap 06
# ============================================================

checklist_path = REPORT_DIR / "stage06_output_checklist.csv"

expected_outputs = [
    PROJECT_ROOT / "notebooks" / "06_chatbot_pipeline_integration.ipynb",
    REPORT_DIR / "chatbot_test_results_stage06.csv",
    REPORT_DIR / "chatbot_sample_responses_stage06.md",
    REPORT_DIR / "stage06_summary.txt",
]

checklist_rows = []

for path in expected_outputs:
    checklist_rows.append({
        "file": str(path.relative_to(PROJECT_ROOT)) if path.exists() else str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0
    })

# Simpan checklist awal
checklist_df = pd.DataFrame(checklist_rows)
checklist_df.to_csv(checklist_path, index=False, encoding="utf-8-sig")

# Tambahkan file checklist itu sendiri setelah berhasil dibuat
checklist_rows.append({
    "file": str(checklist_path.relative_to(PROJECT_ROOT)),
    "exists": checklist_path.exists(),
    "size_bytes": checklist_path.stat().st_size if checklist_path.exists() else 0
})

checklist_df = pd.DataFrame(checklist_rows)
checklist_df.to_csv(checklist_path, index=False, encoding="utf-8-sig")

display(checklist_df)
print("Checklist disimpan ke:", checklist_path)

,file,exists,size_bytes
0,notebooks\06_chatbot_pipeline_integration.ipynb,True,87567
1,reports\stage06\chatbot_test_results_stage06.csv,True,8667
2,reports\stage06\chatbot_sample_responses_stage...,True,8211
3,reports\stage06\stage06_summary.txt,True,1675
4,reports\stage06\stage06_output_checklist.csv,True,257


Checklist disimpan ke: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage06\stage06_output_checklist.csv


### Interpretasi Checklist

Checklist digunakan untuk memastikan bahwa output utama Tahap 06 sudah terbentuk sebelum dilakukan commit dan push ke GitHub.

File minimum yang sebaiknya tersedia:

1. `notebooks/06_chatbot_pipeline_integration.ipynb`
2. `reports/stage06/chatbot_test_results_stage06.csv`
3. `reports/stage06/chatbot_sample_responses_stage06.md`
4. `reports/stage06/stage06_summary.txt`
5. `reports/stage06/stage06_output_checklist.csv`

Jika ada file yang bernilai `exists = False`, simpan notebook terlebih dahulu atau ulangi cell penyimpanan report.

## 17. Kesimpulan Tahap 06

Berdasarkan hasil integrasi, Tahap 06 berhasil membangun pipeline chatbot sederhana yang menghubungkan intent classification dan recommender system. Chatbot sudah dapat menangani beberapa kategori pertanyaan, yaitu sapaan, rekomendasi program studi, roadmap belajar, prospek karier, skill awal, info program studi, klarifikasi minat, dan fallback.

Keterbatasan utama tahap ini adalah jumlah dataset yang masih kecil sehingga model intent classification dan rekomendasi masih perlu diperkuat. Pada tahap berikutnya, sistem dapat dikembangkan menjadi aplikasi Streamlit sederhana agar dapat diuji oleh user secara interaktif.